<a href="https://colab.research.google.com/github/cvbrandoe/DECIDON-NEL/blob/main/Decidon_NEL_explorations_GT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploration des données de vérité terrain — Named Entity Linking

Ce notebook calcule :
- le nombre de valeurs vides (NULL/NIL) dans le tableur
- le nombre de `check_validity` à `True`
- les changements entre `wikidata_url` (avant) et `wikidata_url_valide` (après), selon `vote_result`
- une synthèse des informations des entités bien resolues issues de Wikidata

## Chargement et fonctions utilitaires

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Fonctions utilitaires ---

VALEURS_VIDES = {"", "NULL", "NIL", "NAN", "NONE", "N/A"}

def is_empty_value(x):
    """True si la valeur est NaN/None ou une chaîne représentant du vide (NULL, NIL, ...)."""
    if pd.isna(x):
        return True
    return str(x).strip().upper() in VALEURS_VIDES

def to_bool(x):
    """Convertit des valeurs hétérogènes (bool, str, 0/1) en booléen, NaN sinon."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    s = str(x).strip().lower()
    if s in {"true", "1", "vrai", "yes", "oui"}:
        return True
    if s in {"false", "0", "faux", "no", "non"}:
        return False
    return np.nan

def normalize_url(x):
    """Renvoie None si vide, sinon la chaîne nettoyée (espaces, casse conservée)."""
    return None if is_empty_value(x) else str(x).strip()

In [ ]:
import pandas as pd
import numpy as np

# Fichier CSV séparé par des tabulations
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/DECIDON/GT_analysis/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked_consolidated_intra-seance - nel_consolidated_intra.tsv", sep="\t")

# Colonnes à conserver (les autres sont ignorées pour la suite)
colonnes_utiles = [
    "uuid", "classe", "entity", "span", "start_for_tri", "contexte",
    "task_id", "annotation_id", "seance", "date", "source", "id_entity",
    "check_validity", "wikidata_url_valide", "wikidata_url", "vote", "vote_result",
    "per_fct_per_span", "per_fct_fct_span", "per_fct_image_url", "per_fct_chamber",
    "per_fct_date", "per_fct_match_method", "per_fct_match_score",
    "persons_1_wikidata_url", "persons_2_wikidata_url",
]

# On ne garde que celles réellement présentes dans le fichier (évite un KeyError
# si une colonne comme persons_3_wikidata_url n'existe pas dans cet export)
colonnes_presentes = [c for c in colonnes_utiles if c in df.columns]
colonnes_manquantes = [c for c in colonnes_utiles if c not in df.columns]

if colonnes_manquantes:
    print("Colonnes demandées mais absentes du fichier :", colonnes_manquantes)

df = df[colonnes_presentes]

print(f"Colonnes conservées ({len(df.columns)}) :")
print(list(df.columns))

# Remplacer les valeurs vides de la colonne wikidata_url par "NIL"
df["wikidata_url"] = df["wikidata_url"].apply(lambda x: "NIL" if is_empty_value(x) else x)

# Vérification rapide
print(df["wikidata_url"].value_counts(dropna=False).head())
print(f"Nombre de 'NIL' : {(df['wikidata_url'] == 'NIL').sum()}")

print(f"Lignes : {len(df)}  |  Colonnes : {len(df.columns)}")
df.head()

Colonnes conservées (26) :
['uuid', 'classe', 'entity', 'span', 'start_for_tri', 'contexte', 'task_id', 'annotation_id', 'seance', 'date', 'source', 'id_entity', 'check_validity', 'wikidata_url_valide', 'wikidata_url', 'vote', 'vote_result', 'per_fct_per_span', 'per_fct_fct_span', 'per_fct_image_url', 'per_fct_chamber', 'per_fct_date', 'per_fct_match_method', 'per_fct_match_score', 'persons_1_wikidata_url', 'persons_2_wikidata_url']
wikidata_url
NIL                                       1611
https://www.wikidata.org/wiki/Q3132698     220
https://www.wikidata.org/wiki/Q3171692     121
https://www.wikidata.org/wiki/Q2978569     106
https://www.wikidata.org/wiki/Q2935008      73
Name: count, dtype: int64
Nombre de 'NIL' : 1611
Lignes : 4686  |  Colonnes : 26


/tmp/ipykernel_1743/1897587432.py:5: DtypeWarning: Columns (60,71,93,104,115,126,147,148,150,151,152,153,158,159,161,162,163,164) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/DECIDON/GT_analysis/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked_consolidated_intra-seance - nel_consolidated_intra.tsv", sep="\t")


,uuid,classe,entity,span,start_for_tri,contexte,task_id,annotation_id,seance,date,...,vote_result,per_fct_per_span,per_fct_fct_span,per_fct_image_url,per_fct_chamber,per_fct_date,per_fct_match_method,per_fct_match_score,persons_1_wikidata_url,persons_2_wikidata_url
0,2ab5d97d-8316-4cff-9325-de0173bdc86a,PER,Henri Brisson,600:613,600,fr.; 2^nd^ Du projet de loi tendant à établir ...,109,162,deputes,1890-06-24,...,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.wikidata.org/wiki/Q669725,https://www.wikidata.org/wiki/Q669725
1,33b67fa6-7fce-49d8-977b-f84348b40a7d,PER,le ministre des affaires étrangères,620:655,620,de loi tendant à établir d'office sur la commu...,109,162,deputes,1890-06-24,...,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.wikidata.org/wiki/Q358387,NaN
2,834db6cd-29be-4049-ae0f-4fe661a1e5ee,PER,Henri Brisson,662:675,662,la commune de Saint-Cirq (Aveyron) une contrib...,109,162,deputes,1890-06-24,...,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.wikidata.org/wiki/Q669725,https://www.wikidata.org/wiki/Q669725
3,5c6a5a4c-0735-4025-af6d-f5d934e54efc,PER,le ministre des affaires étrangères,677:712,677,de Saint-Cirq (Aveyron) une contribution extra...,109,162,deputes,1890-06-24,...,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.wikidata.org/wiki/Q358387,NaN
4,1f743b04-7d39-4cda-bf51-3fe02de9cee9,PER,du Breuil de Saint-Germain,752:778,752,loi tendant à établir d'office une imposition ...,109,162,deputes,1890-06-24,...,ambiguous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.wikidata.org/wiki/Q16026138,https://www.wikidata.org/wiki/Q16026138


## Valeurs vides (NULL / NIL)

In [ ]:
# Nombre de valeurs vides par colonne
empty_per_col = df.applymap(is_empty_value).sum().sort_values(ascending=False)
print("Valeurs vides par colonne :")
print(empty_per_col)

Valeurs vides par colonne :
per_fct_chamber           4556
per_fct_per_span          4556
per_fct_match_method      4556
per_fct_image_url         4556
per_fct_fct_span          4556
per_fct_date              4556
per_fct_match_score       4556
id_entity                 1611
wikidata_url              1611
vote                      1611
persons_2_wikidata_url    1212
persons_1_wikidata_url     197
wikidata_url_valide        176
contexte                     1
date                         0
span                         0
entity                       0
classe                       0
uuid                         0
seance                       0
annotation_id                0
task_id                      0
start_for_tri                0
vote_result                  0
source                       0
check_validity               0
dtype: int64


/tmp/ipykernel_1743/2387185385.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  empty_per_col = df.applymap(is_empty_value).sum().sort_values(ascending=False)


In [ ]:
# Focus sur les colonnes prioritaires (celles marquées d'une étoile)
colonnes_cles = ["entity", "date", "check_validity", "wikidata_url_valide", "wikidata_url", "vote_result"]
empty_cles = df[colonnes_cles].applymap(is_empty_value).sum()
print(empty_cles)

entity                    0
date                      0
check_validity            0
wikidata_url_valide     176
wikidata_url           1611
vote_result               0
dtype: int64


/tmp/ipykernel_1743/3108322881.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  empty_cles = df[colonnes_cles].applymap(is_empty_value).sum()


In [ ]:
colonnes_url = ["wikidata_url_valide", "wikidata_url"]

def empty_counts_by(df, group_col, value_cols):
    """Nombre et % de valeurs vides pour value_cols, par modalité de group_col."""
    resultats = {}
    for col in value_cols:
        empty_flag = df[col].apply(is_empty_value)
        counts = (
            empty_flag.groupby(df[group_col], dropna=False)
            .agg(nb_vides="sum", nb_total="count")
        )
        counts["pct_vides"] = (counts["nb_vides"] / counts["nb_total"]).round(4)
        resultats[col] = counts
    return resultats

# --- Par classe ---
resultats_classe = empty_counts_by(df, "classe", colonnes_url)
for col, tableau in resultats_classe.items():
    print(f"\n=== Vides pour '{col}', par classe ===")
    print(tableau)


=== Vides pour 'wikidata_url_valide', par classe ===
        nb_vides  nb_total  pct_vides
classe                               
PER          167      2877      0.058
SPK            9      1809      0.005

=== Vides pour 'wikidata_url', par classe ===
        nb_vides  nb_total  pct_vides
classe                               
PER          862      2877     0.2996
SPK          749      1809     0.4140


## Valeurs non vides

In [ ]:
# Nombre de valeurs NON vides par colonne
nonempty_per_col = df.apply(lambda col: (~col.apply(is_empty_value)).sum())
print("Valeurs non vides par colonne :")
print(nonempty_per_col)

Valeurs non vides par colonne :
uuid                      4686
classe                    4686
entity                    4686
span                      4686
start_for_tri             4686
contexte                  4685
task_id                   4686
annotation_id             4686
seance                    4686
date                      4686
source                    4686
id_entity                 3075
check_validity            4686
wikidata_url_valide       4510
wikidata_url              3075
vote                      3075
vote_result               4686
per_fct_per_span           130
per_fct_fct_span           130
per_fct_image_url          130
per_fct_chamber            130
per_fct_date               130
per_fct_match_method       130
per_fct_match_score        130
persons_1_wikidata_url    4489
persons_2_wikidata_url    3474
dtype: int64


## Valeurs les plus frequentes

In [ ]:
print(df["wikidata_url_valide"].value_counts(dropna=False).head(10))

wikidata_url_valide
https://www.wikidata.org/wiki/Q3132698    256
NIL                                       174
https://www.wikidata.org/wiki/Q459510     133
https://www.wikidata.org/wiki/Q2978569    127
https://www.wikidata.org/wiki/Q296673     123
https://www.wikidata.org/wiki/Q3143189    122
https://www.wikidata.org/wiki/Q3171692    121
https://www.wikidata.org/wiki/Q471311      92
https://www.wikidata.org/wiki/Q1063452     74
https://www.wikidata.org/wiki/Q2935008     73
Name: count, dtype: int64


In [ ]:
import requests
import re
import time

def extraire_qid(url):
    """Extrait le QID (ex: Q12345) depuis une URL Wikidata."""
    if pd.isna(url) or not isinstance(url, str):
        return None
    match = re.search(r"(Q\d+)", url)
    return match.group(1) if match else None

def get_prenom_nom(qids, batch_size=100):
    """Interroge le point SPARQL Wikidata pour obtenir prénom(s) et nom(s) de famille."""
    endpoint = "https://query.wikidata.org/sparql"
    headers = {"User-Agent": "notebook-nel-exploration/1.0", "Accept": "application/json"}
    resultats = {}

    for i in range(0, len(qids), batch_size):
        lot = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in lot)
        query = f"""
        SELECT ?person ?personLabel ?prenomLabel ?nomLabel WHERE {{
          VALUES ?person {{ {values} }}
          OPTIONAL {{ ?person wdt:P735 ?prenom. }}
          OPTIONAL {{ ?person wdt:P734 ?nom. }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "fr,en". }}
        }}
        """
        r = requests.get(endpoint, params={"query": query, "format": "json"}, headers=headers, timeout=60)
        bindings = r.json()["results"]["bindings"]
        for b in bindings:
            qid = b["person"]["value"].split("/")[-1]
            resultats.setdefault(qid, {"nom_complet": None, "prenoms": set(), "noms": set()})
            resultats[qid]["nom_complet"] = b.get("personLabel", {}).get("value")
            if "prenomLabel" in b:
                resultats[qid]["prenoms"].add(b["prenomLabel"]["value"])
            if "nomLabel" in b:
                resultats[qid]["noms"].add(b["nomLabel"]["value"])
        time.sleep(0.5)

    # Mise en forme finale
    lignes = []
    for qid, v in resultats.items():
        lignes.append({
            "qid": qid,
            "nom_complet": v["nom_complet"],
            "prenom": ", ".join(sorted(v["prenoms"])) if v["prenoms"] else None,
            "nom": ", ".join(sorted(v["noms"])) if v["noms"] else None,
        })
    return pd.DataFrame(lignes)


top20 = df["wikidata_url_valide"].value_counts(dropna=False).head(20)

# Extraction des QID à partir de ces 20 URI (on ignore "NIL"/vides éventuels)
top20_urls = [u for u in top20.index if not is_empty_value(u)]
qids_top20 = [extraire_qid(u) for u in top20_urls]
qids_top20 = [q for q in qids_top20 if q is not None]

print(f"{len(qids_top20)} QID à interroger")
print(qids_top20)

# Récupération prénom / nom via SPARQL (fonction définie précédemment)
df_personnes_top20 = get_prenom_nom(qids_top20)
df_personnes_top20

resume_top20 = pd.DataFrame({
    "wikidata_url": top20_urls,
    "qid": qids_top20,
    "nb_occurrences": top20.loc[top20_urls].values
})
resume_top20 = resume_top20.merge(df_personnes_top20, on="qid", how="left")
resume_top20 = resume_top20.sort_values("nb_occurrences", ascending=False)
resume_top20

19 QID à interroger
['Q3132698', 'Q459510', 'Q2978569', 'Q296673', 'Q3143189', 'Q3171692', 'Q471311', 'Q1063452', 'Q2935008', 'Q585025', 'Q158772', 'Q3184755', 'Q3121364', 'Q2861802', 'Q1934639', 'Q2899637', 'Q317957', 'Q3385710', 'Q3188688']


,wikidata_url,qid,nb_occurrences,nom_complet,prenom,nom
0,https://www.wikidata.org/wiki/Q3132698,Q3132698,256,Henry Chéron,"Henri, Henry",Chéron
1,https://www.wikidata.org/wiki/Q459510,Q459510,133,Fernand Bouisson,Fernand,Bouisson
2,https://www.wikidata.org/wiki/Q2978569,Q2978569,127,Claudius Gallet,Claudius,Gallet
3,https://www.wikidata.org/wiki/Q296673,Q296673,123,Armand Fallières,"Armand, Clément",Fallières
4,https://www.wikidata.org/wiki/Q3143189,Q3143189,122,Humbert Ricolfi,Humbert,Ricolfi
5,https://www.wikidata.org/wiki/Q3171692,Q3171692,121,Jean Desbons,Jean,Desbons
6,https://www.wikidata.org/wiki/Q471311,Q471311,92,Charles Floquet,Charles,Floquet
7,https://www.wikidata.org/wiki/Q1063452,Q1063452,74,Charles-Émile Freppel,Charles-Émile,None
8,https://www.wikidata.org/wiki/Q2935008,Q2935008,73,Camille Margaine,Camille,Margaine
9,https://www.wikidata.org/wiki/Q585025,Q585025,70,Philippe Le Royer,"Philippe, Élie",Le Roy


In [ ]:
def get_occupation_sycomore(qids, batch_size=100):
    """Interroge Wikidata pour obtenir les occupations (P106) et l'identifiant Sycomore (P1045)."""
    endpoint = "https://query.wikidata.org/sparql"
    headers = {"User-Agent": "notebook-nel-exploration/1.0", "Accept": "application/json"}
    resultats = {}

    for i in range(0, len(qids), batch_size):
        lot = qids[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in lot)
        query = f"""
        SELECT ?person ?occupationLabel ?sycomoreId WHERE {{
          VALUES ?person {{ {values} }}
          OPTIONAL {{ ?person wdt:P106 ?occupation. }}
          OPTIONAL {{ ?person wdt:P1045 ?sycomoreId. }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "fr,en". }}
        }}
        """
        r = requests.get(endpoint, params={"query": query, "format": "json"}, headers=headers, timeout=60)
        bindings = r.json()["results"]["bindings"]
        for b in bindings:
            qid = b["person"]["value"].split("/")[-1]
            resultats.setdefault(qid, {"occupations": set(), "sycomore_ids": set()})
            if "occupationLabel" in b:
                resultats[qid]["occupations"].add(b["occupationLabel"]["value"])
            if "sycomoreId" in b:
                resultats[qid]["sycomore_ids"].add(b["sycomoreId"]["value"])
        time.sleep(0.5)

    lignes = []
    for qid, v in resultats.items():
        lignes.append({
            "qid": qid,
            "occupation": ", ".join(sorted(v["occupations"])) if v["occupations"] else None,
            "sycomore_id": ", ".join(sorted(v["sycomore_ids"])) if v["sycomore_ids"] else None,
            "a_sycomore": bool(v["sycomore_ids"]),
        })
    return pd.DataFrame(lignes)


df_occ_syc_top20 = get_occupation_sycomore(qids_top20)
df_occ_syc_top20

,qid,occupation,sycomore_id,a_sycomore
0,Q158772,"avocat ou avocate, chef de gouvernement, juris...",285,True
1,Q296673,"avocat ou avocate, juriste, personnalité polit...",2901,True
2,Q317957,"fonctionnaire, ingénieur ou ingénieure, person...",None,False
3,Q459510,personnalité politique,1043,True
4,Q471311,"fonctionnaire, juriste, personnalité politique",3032,True
5,Q585025,"avocat ou avocate, personnalité politique",8640,True
6,Q1063452,"personnalité politique, prêtre catholique, thé...",3148,True
7,Q1934639,"diplomate, juriste, personnalité politique",1942,True
8,Q2861802,"avocat ou avocate, personnalité politique",1816,True
9,Q2899637,"juriste, ministre, parlementaire, personnalité...",5569,True


In [ ]:
# Fusion avec le résumé existant
resume_top20 = resume_top20.merge(df_occ_syc_top20, on="qid", how="left")
resume_top20 = resume_top20.sort_values("nb_occurrences", ascending=False)
resume_top20

,wikidata_url,qid,nb_occurrences,nom_complet,prenom,nom,occupation,sycomore_id,a_sycomore
0,https://www.wikidata.org/wiki/Q3132698,Q3132698,256,Henry Chéron,"Henri, Henry",Chéron,"avocat ou avocate, personnalité politique",1768,True
1,https://www.wikidata.org/wiki/Q459510,Q459510,133,Fernand Bouisson,Fernand,Bouisson,personnalité politique,1043,True
2,https://www.wikidata.org/wiki/Q2978569,Q2978569,127,Claudius Gallet,Claudius,Gallet,"médecin, personnalité politique",None,False
3,https://www.wikidata.org/wiki/Q296673,Q296673,123,Armand Fallières,"Armand, Clément",Fallières,"avocat ou avocate, juriste, personnalité polit...",2901,True
4,https://www.wikidata.org/wiki/Q3143189,Q3143189,122,Humbert Ricolfi,Humbert,Ricolfi,personnalité politique,6752,True
5,https://www.wikidata.org/wiki/Q3171692,Q3171692,121,Jean Desbons,Jean,Desbons,personnalité politique,2415,True
6,https://www.wikidata.org/wiki/Q471311,Q471311,92,Charles Floquet,Charles,Floquet,"fonctionnaire, juriste, personnalité politique",3032,True
7,https://www.wikidata.org/wiki/Q1063452,Q1063452,74,Charles-Émile Freppel,Charles-Émile,None,"personnalité politique, prêtre catholique, thé...",3148,True
8,https://www.wikidata.org/wiki/Q2935008,Q2935008,73,Camille Margaine,Camille,Margaine,personnalité politique,7999,True
9,https://www.wikidata.org/wiki/Q585025,Q585025,70,Philippe Le Royer,"Philippe, Élie",Le Roy,"avocat ou avocate, personnalité politique",8640,True


## Valeurs ponderées

In [ ]:
# Extraction de tous les QID uniques présents dans wikidata_url_valide
tous_les_qids = (
    df["wikidata_url_valide"]
    .apply(extraire_qid)
    .dropna()
    .unique()
    .tolist()
)

print(f"{len(tous_les_qids)} QID uniques à interroger (ensemble du corpus)")

992 QID uniques à interroger (ensemble du corpus)


In [ ]:
# Récupération occupation (P106) + Sycomore (P1045) pour tous les QID
# ⚠️ Peut prendre du temps si le nombre de QID est élevé (batch_size=100, pause 0.5s entre lots)
df_occ_syc_full = get_occupation_sycomore(tous_les_qids, batch_size=100)

print(f"Réponses obtenues pour {len(df_occ_syc_full)} / {len(tous_les_qids)} QID")
df_occ_syc_full

Réponses obtenues pour 992 / 992 QID


,qid,occupation,sycomore_id,a_sycomore
0,Q535,"artiste graphique, auteur, chroniqueur de voya...",8795,True
1,Q171730,"auteur, chef de gouvernement, enseignant ou en...",1844,True
2,Q179491,personnalité politique,5873,True
3,Q215778,"fonctionnaire, ingénieur civil, ingénieur ou i...",1453,True
4,Q217706,"diplomate, fonctionnaire, juge, juriste, perso...",1083,True
...,...,...,...,...
987,Q3557514,personnalité politique,8069,True
988,Q3559439,personnalité politique,None,False
989,Q3579829,"botaniste, mycologue, personnalité politique",None,False
990,Q3588533,"avocat ou avocate, fonctionnaire, personnalité...",8982,True


In [ ]:
# Nombre d'occurrences de chaque QID dans le corpus (pondération par mention, pas par entité unique)
occurrences_qid = (
    df["wikidata_url_valide"]
    .apply(extraire_qid)
    .dropna()
    .value_counts()
    .rename_axis("qid")
    .reset_index(name="nb_occurrences")
)

# Fusion : une ligne par entité unique, avec son nombre d'occurrences dans le corpus
synthese_entites = occurrences_qid.merge(df_occ_syc_full, on="qid", how="left")
synthese_entites = synthese_entites.sort_values("nb_occurrences", ascending=False)
synthese_entites

,qid,nb_occurrences,occupation,sycomore_id,a_sycomore
0,Q3132698,256,"avocat ou avocate, personnalité politique",1768,True
1,Q459510,133,personnalité politique,1043,True
2,Q2978569,127,"médecin, personnalité politique",None,False
3,Q296673,123,"avocat ou avocate, juriste, personnalité polit...",2901,True
4,Q3143189,122,personnalité politique,6752,True
...,...,...,...,...,...
372,Q215778,1,"fonctionnaire, ingénieur civil, ingénieur ou i...",1453,True
991,Q3384693,1,personnalité politique,2353,True
990,Q16025566,1,personnalité politique,2385,True
989,Q2847076,1,personnalité politique,"2395, 8209",True


In [ ]:
nb_entites_total = len(synthese_entites)
nb_avec_sycomore = synthese_entites["a_sycomore"].sum()

print(f"Entités uniques avec un lien Sycomore : {nb_avec_sycomore} / {nb_entites_total} "
      f"({nb_avec_sycomore/nb_entites_total:.2%})")

# Idem, mais pondéré par le nombre de mentions dans le corpus (pas juste par entité unique)
mentions_avec_sycomore = synthese_entites.loc[synthese_entites["a_sycomore"], "nb_occurrences"].sum()
mentions_total = synthese_entites["nb_occurrences"].sum()

print(f"Mentions liées à une entité avec Sycomore : {mentions_avec_sycomore} / {mentions_total} "
      f"({mentions_avec_sycomore/mentions_total:.2%})")

Entités uniques avec un lien Sycomore : 819 / 992 (82.56%)
Mentions liées à une entité avec Sycomore : 3750 / 4510 (83.15%)


In [ ]:
# Par entité unique (une occupation comptée une fois par personne, même si mentionnée plusieurs fois)
occupations_par_entite = (
    synthese_entites["occupation"]
    .dropna()
    .str.split(", ")
    .explode()
    .value_counts()
)
print("Occupations les plus fréquentes (par entité unique) :")
print(occupations_par_entite.head(30))

Occupations les plus fréquentes (par entité unique) :
occupation
personnalité politique                                 985
avocat ou avocate                                       90
journaliste                                             63
fonctionnaire                                           44
diplomate                                               38
juriste                                                 33
médecin                                                 33
écrivain ou écrivaine                                   25
militaire                                               20
officier                                                16
ministre                                                15
personnalité du monde des affaires                      13
industriel ou industrielle                              12
magistrat ou magistrate                                 11
ingénieur ou ingénieure                                 10
poète ou poétesse                                 

In [ ]:
# Par mention dans le corpus (pondéré par nb_occurrences : une occupation compte autant de fois
# que l'entité apparaît dans le corpus)
occ_ponderees = synthese_entites.dropna(subset=["occupation"]).copy()
occ_ponderees["occupation"] = occ_ponderees["occupation"].str.split(", ")
occ_ponderees = occ_ponderees.explode("occupation")

occupations_par_mention = (
    occ_ponderees.groupby("occupation")["nb_occurrences"]
    .sum()
    .sort_values(ascending=False)
)
print("Occupations les plus fréquentes (pondérées par nombre de mentions) :")
print(occupations_par_mention.head(30))

Occupations les plus fréquentes (pondérées par nombre de mentions) :
occupation
personnalité politique                                 4496
avocat ou avocate                                      1042
juriste                                                 516
fonctionnaire                                           252
médecin                                                 218
journaliste                                             206
diplomate                                               190
ministre                                                141
chef de gouvernement                                    122
officier                                                 95
parlementaire                                            95
écrivain ou écrivaine                                    90
ingénieur ou ingénieure                                  84
prêtre catholique                                        75
théologien ou théologienne catholique                    74
évêque catholique   

In [ ]:
recap_synthese = pd.DataFrame({
    "indicateur": [
        "Entités uniques interrogées",
        "Entités uniques avec Sycomore",
        "Entités uniques sans occupation renseignée",
        "Mentions totales dans le corpus",
        "Mentions liées à une entité avec Sycomore",
    ],
    "valeur": [
        nb_entites_total,
        nb_avec_sycomore,
        synthese_entites["occupation"].isna().sum(),
        mentions_total,
        mentions_avec_sycomore,
    ]
})
recap_synthese

,indicateur,valeur
0,Entités uniques interrogées,992
1,Entités uniques avec Sycomore,819
2,Entités uniques sans occupation renseignée,1
3,Mentions totales dans le corpus,4510
4,Mentions liées à une entité avec Sycomore,3750


## Les entites bien résolues dès le départ automatiquement

In [ ]:
# Lignes où l'URI n'a PAS changé (avant = après), en excluant les cas où les deux sont vides
uri_corrects = df[
    (~df["url_changed"]) & (df["url_apres_norm"].notna())
]

print(f"Nombre d'URI corrects (inchangés, non vides) : {len(uri_corrects)} / {len(df)}")

colonnes_affichage = ["uuid", "entity", "classe", "vote_result", "wikidata_url", "wikidata_url_valide"]
colonnes_affichage = [c for c in colonnes_affichage if c in df.columns]
uri_corrects[colonnes_affichage]

uri_corrects_uniques = uri_corrects["wikidata_url_valide"].unique()
print(f"URI uniques corrects : {len(uri_corrects_uniques)}")
print(uri_corrects_uniques)

Nombre d'URI corrects (inchangés, non vides) : 3009 / 4686
URI uniques corrects : 827
['https://www.wikidata.org/wiki/Q669725'
 'https://www.wikidata.org/wiki/Q358387'
 'https://www.wikidata.org/wiki/Q16026138'
 'https://www.wikidata.org/wiki/Q1934639'
 'https://www.wikidata.org/wiki/Q15969591'
 'https://www.wikidata.org/wiki/Q1063452'
 'https://www.wikidata.org/wiki/Q217706'
 'https://www.wikidata.org/wiki/Q3121364'
 'https://www.wikidata.org/wiki/Q3121310'
 'https://www.wikidata.org/wiki/Q2853848'
 'https://www.wikidata.org/wiki/Q3172111'
 'https://www.wikidata.org/wiki/Q3188598'
 'https://www.wikidata.org/wiki/Q3387297'
 'https://www.wikidata.org/wiki/Q8343994'
 'https://www.wikidata.org/wiki/Q17629005'
 'https://www.wikidata.org/wiki/Q317957'
 'https://www.wikidata.org/wiki/Q2862011'
 'https://www.wikidata.org/wiki/Q459914'
 'https://www.wikidata.org/wiki/Q2835279'
 'https://www.wikidata.org/wiki/Q16026099'
 'https://www.wikidata.org/wiki/Q16026549'
 'https://www.wikidata.org/wiki/

In [ ]:
# Top 20 des URI corrects (inchangés, non vides) par fréquence
top20_uri_corrects = uri_corrects["wikidata_url_valide"].value_counts().head(20)
print(top20_uri_corrects)

wikidata_url_valide
https://www.wikidata.org/wiki/Q3132698     220
https://www.wikidata.org/wiki/Q3171692     121
https://www.wikidata.org/wiki/Q2978569     106
https://www.wikidata.org/wiki/Q2935008      73
https://www.wikidata.org/wiki/Q1063452      67
https://www.wikidata.org/wiki/Q3121364      59
https://www.wikidata.org/wiki/Q2861802      55
https://www.wikidata.org/wiki/Q158772       51
https://www.wikidata.org/wiki/Q317957       37
https://www.wikidata.org/wiki/Q3174745      35
https://www.wikidata.org/wiki/Q16025678     34
https://www.wikidata.org/wiki/Q3103471      33
https://www.wikidata.org/wiki/Q3093994      33
https://www.wikidata.org/wiki/Q16026138     28
https://www.wikidata.org/wiki/Q3052727      27
https://www.wikidata.org/wiki/Q2835279      27
https://www.wikidata.org/wiki/Q3102255      27
https://www.wikidata.org/wiki/Q3035090      25
https://www.wikidata.org/wiki/Q15969591     24
https://www.wikidata.org/wiki/Q3184755      24
Name: count, dtype: int64


In [ ]:
top20_details = uri_corrects[
    uri_corrects["wikidata_url_valide"].isin(top20_uri_corrects.index)
][["wikidata_url_valide", "classe", "entity"]].drop_duplicates()

top20_details = top20_details.merge(
    top20_uri_corrects.rename("nb_occurrences"),
    left_on="wikidata_url_valide", right_index=True
).sort_values("nb_occurrences", ascending=False)

top20_details

,wikidata_url_valide,classe,entity,nb_occurrences
1544,https://www.wikidata.org/wiki/Q3132698,SPK,le ministre des finances,220
1453,https://www.wikidata.org/wiki/Q3132698,PER,Henry Chéron,220
1460,https://www.wikidata.org/wiki/Q3132698,PER,le ministre des finances,220
1539,https://www.wikidata.org/wiki/Q3132698,SPK,Henry Chéron,220
1793,https://www.wikidata.org/wiki/Q3132698,PER,du ministre des finances,220
...,...,...,...,...
7,https://www.wikidata.org/wiki/Q15969591,PER,le comte Lemercier,24
293,https://www.wikidata.org/wiki/Q15969591,PER,Lemercier,24
3339,https://www.wikidata.org/wiki/Q3184755,PER,le ministre de l'instruction publique,24
3104,https://www.wikidata.org/wiki/Q3184755,SPK,le ministre de l'instruction publique,24


In [ ]:
# Extraction des QID à partir des URI du top 20 (déduplication)
top20_details["qid"] = top20_details["wikidata_url_valide"].apply(extraire_qid)
qids_top20_corrects = top20_details["qid"].dropna().unique().tolist()

print(f"{len(qids_top20_corrects)} QID à interroger")
print(qids_top20_corrects)

# Récupération prénom / nom via SPARQL (fonction définie précédemment)
df_personnes_top20_corrects = get_prenom_nom(qids_top20_corrects)
df_personnes_top20_corrects


# Fusion avec le détail classe/entity/occurrences
resultat_final = top20_details.merge(df_personnes_top20_corrects, on="qid", how="left")
resultat_final = resultat_final.sort_values("nb_occurrences", ascending=False)
resultat_final

20 QID à interroger
['Q3132698', 'Q3171692', 'Q2978569', 'Q2935008', 'Q1063452', 'Q3121364', 'Q2861802', 'Q158772', 'Q317957', 'Q3174745', 'Q16025678', 'Q3093994', 'Q3103471', 'Q16026138', 'Q3102255', 'Q3052727', 'Q2835279', 'Q3035090', 'Q15969591', 'Q3184755']


,wikidata_url_valide,classe,entity,nb_occurrences,qid,nom_complet,prenom,nom
0,https://www.wikidata.org/wiki/Q3132698,SPK,le ministre des finances,220,Q3132698,Henry Chéron,"Henri, Henry",Chéron
1,https://www.wikidata.org/wiki/Q3132698,PER,Henry Chéron,220,Q3132698,Henry Chéron,"Henri, Henry",Chéron
2,https://www.wikidata.org/wiki/Q3132698,PER,le ministre des finances,220,Q3132698,Henry Chéron,"Henri, Henry",Chéron
3,https://www.wikidata.org/wiki/Q3132698,SPK,Henry Chéron,220,Q3132698,Henry Chéron,"Henri, Henry",Chéron
4,https://www.wikidata.org/wiki/Q3132698,PER,du ministre des finances,220,Q3132698,Henry Chéron,"Henri, Henry",Chéron
...,...,...,...,...,...,...,...,...
58,https://www.wikidata.org/wiki/Q15969591,PER,le comte Lemercier,24,Q15969591,Anatole Lemercier,Anatole,Lemercier
59,https://www.wikidata.org/wiki/Q15969591,PER,Lemercier,24,Q15969591,Anatole Lemercier,Anatole,Lemercier
60,https://www.wikidata.org/wiki/Q3184755,PER,le ministre de l'instruction publique,24,Q3184755,Joseph Chaumié,Joseph,Chaumié
61,https://www.wikidata.org/wiki/Q3184755,SPK,le ministre de l'instruction publique,24,Q3184755,Joseph Chaumié,Joseph,Chaumié


## Les entités hapax

In [ ]:
# Identifiants (wikidata_url_valide) qui apparaissent une seule fois
occurrences = df["wikidata_url_valide"].value_counts(dropna=False)
uri_uniques = occurrences[occurrences == 1]

print(f"Nombre d'URI apparaissant une seule fois : {len(uri_uniques)}")

# Les 10 premières
top10_uniques = uri_uniques.head(20)
print(top10_uniques)

Nombre d'URI apparaissant une seule fois : 622
wikidata_url_valide
https://www.wikidata.org/wiki/Q8442        1
https://www.wikidata.org/wiki/Q182701      1
https://www.wikidata.org/wiki/Q15711646    1
https://www.wikidata.org/wiki/Q3092536     1
https://www.wikidata.org/wiki/Q3161144     1
https://www.wikidata.org/wiki/Q3270743     1
https://www.wikidata.org/wiki/Q3419119     1
https://www.wikidata.org/wiki/Q16025566    1
https://www.wikidata.org/wiki/Q2847076     1
https://www.wikidata.org/wiki/Q3056924     1
https://www.wikidata.org/wiki/Q16025572    1
https://www.wikidata.org/wiki/Q309995      1
https://www.wikidata.org/wiki/Q939193      1
https://www.wikidata.org/wiki/Q8306562     1
https://www.wikidata.org/wiki/Q15302973    1
https://www.wikidata.org/wiki/Q2960028     1
https://www.wikidata.org/wiki/Q15974106    1
https://www.wikidata.org/wiki/Q15974110    1
https://www.wikidata.org/wiki/Q3132908     1
https://www.wikidata.org/wiki/Q15973888    1
Name: count, dtype: int64


## Les mentions non resolues

In [ ]:
# Mentions (classe + entity) dont wikidata_url_valide est vide
mentions_sans_url = df.loc[df["wikidata_url_valide"].apply(is_empty_value), ["classe", "entity"]]

print(f"Nombre de mentions sans wikidata_url_valide : {len(mentions_sans_url)}")
mentions_sans_url

Nombre de mentions sans wikidata_url_valide : 176


,classe,entity
34,PER,le marquis de La Ferrière
99,PER,la sœur supé- rieuse
100,PER,le maire
101,PER,le préfet
102,PER,le préfet
...,...,...
4116,PER,Niveaux
4348,PER,le président de la commission
4520,PER,(Marques d'approbation à droite et au centre.)
4583,PER,le ministre de la guerre de 1876


In [ ]:
mentions_sans_url_counts = (
    mentions_sans_url.groupby(["classe", "entity"])
    .size()
    .reset_index(name="nb_occurrences")
    .sort_values("nb_occurrences", ascending=False)
)
mentions_sans_url_counts

,classe,entity,nb_occurrences
108,PER,le préfet de la Haute-Marne,6
25,PER,Gros,5
94,PER,le maire,5
107,PER,le préfet,5
124,SPK,Emile Faure,5
...,...,...,...
121,PER,préfet des Landes,1
123,PER,secrétaire général,1
122,PER,seconde sœur de Vicq,1
125,SPK,Plusieurs voix à gauche.* Lesquelles,1


In [ ]:
# Top 20 des mentions (classe + entity) sans wikidata_url_valide, par fréquence
top20_mentions_sans_url = mentions_sans_url_counts.head(20)
top20_mentions_sans_url

,classe,entity,nb_occurrences
108,PER,le préfet de la Haute-Marne,6
25,PER,Gros,5
94,PER,le maire,5
107,PER,le préfet,5
124,SPK,Emile Faure,5
115,PER,le secrétaire général,4
48,PER,Ortal,3
38,PER,Le maire,3
78,PER,l'inspecteur primaire,3
80,PER,l'institutrice,3


## Changements entre `wikidata_url` (avant) et `wikidata_url_valide` (après), selon `vote_result`

In [ ]:
df["url_avant_norm"] = df["wikidata_url"].apply(normalize_url)
df["url_apres_norm"] = df["wikidata_url_valide"].apply(normalize_url)

# Une ligne est "changée" si la valeur normalisée diffère (y compris apparition/disparition)
df["url_changed"] = df["url_avant_norm"] != df["url_apres_norm"]

total_changed = df["url_changed"].sum()
print(f"Total de lignes où l'URL a changé : {total_changed} / {len(df)} ({total_changed/len(df):.2%})")

Total de lignes où l'URL a changé : 1677 / 4686 (35.79%)


In [ ]:
# Sélectionner les lignes où l'URL a changé entre "avant" et "après"
df["url_avant_norm"] = df["wikidata_url"].apply(normalize_url)
df["url_apres_norm"] = df["wikidata_url_valide"].apply(normalize_url)
df["url_changed"] = df["url_avant_norm"] != df["url_apres_norm"]

lignes_changees = df[df["url_changed"]]

print(f"Nombre de lignes changées : {len(lignes_changees)} / {len(df)}")

# Colonnes utiles pour l'inspection
colonnes_affichage = ["uuid", "entity", "classe", "source", "date", "vote_result",
                       "wikidata_url", "wikidata_url_valide"]
colonnes_affichage = [c for c in colonnes_affichage if c in lignes_changees.columns]

lignes_changees[colonnes_affichage]

Nombre de lignes changées : 1677 / 4686


,uuid,entity,classe,source,date,vote_result,wikidata_url,wikidata_url_valide
24,c935a0f2-a518-4964-9242-34b847464d31,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
25,87c9b2b6-b99e-4d2f-bf6c-b3a3d1e019b7,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
27,74f241d3-3049-4310-ba54-d0d21d32939e,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
31,1a806393-2098-4207-9917-f19bdfda3a7a,Lacroix,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q3120001
33,c35bf626-20fe-4453-8ad6-f5316aa7bab0,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
...,...,...,...,...,...,...,...,...
4673,c2f2e846-5eab-4b6c-8c3e-efae1b33b132,le rapporteur,SPK,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q2861822
4674,014b5df8-fae8-4d4d-83fa-b06e8e637e1a,le général Robert,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q3386817
4675,6f5ed8a1-9905-452b-8b39-8b58dafcdc09,le président,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q585025
4676,b10de028-5bef-4d85-8db0-978a9721d0cf,le président,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q585025


In [ ]:
# Répartition des changements selon vote_result
changes_by_vote = (
    df.groupby("vote_result", dropna=False)["url_changed"]
      .agg(nb_changements="sum", nb_total="count")
)
changes_by_vote["pct_changements"] = (changes_by_vote["nb_changements"] / changes_by_vote["nb_total"]).round(4)

changes_by_vote

,nb_changements,nb_total,pct_changements
vote_result,,,
ambiguous,172,1373,0.1253
intra-seance,0,130,0.0000
resolved_consensus,382,2060,0.1854
resolved_unique,1015,1015,1.0000
unresolved,108,108,1.0000


In [ ]:
# Répartition des changements selon la classe
changes_by_classe = (
    df.groupby("classe", dropna=False)["url_changed"]
      .agg(nb_changements="sum", nb_total="count")
)
changes_by_classe["pct_changements"] = (changes_by_classe["nb_changements"] / changes_by_classe["nb_total"]).round(4)

changes_by_classe

,nb_changements,nb_total,pct_changements
classe,,,
PER,919,2877,0.3194
SPK,758,1809,0.4190


In [ ]:
# Vue croisée détaillée : changé / inchangé x vote_result
crosstab = pd.crosstab(df["vote_result"], df["url_changed"], dropna=False, margins=True)
crosstab

url_changed,False,True,All
vote_result,,,
ambiguous,1201,172,1373
intra-seance,130,0,130
resolved_consensus,1678,382,2060
resolved_unique,0,1015,1015
unresolved,0,108,108
All,3009,1677,4686


### Cas peu ambigus : resolved_consensus et resolved_unique

In [ ]:
groupe_peu_ambigu = ["resolved_consensus", "resolved_unique"]

changements_peu_ambigus = df[
    df["url_changed"] & df["vote_result"].isin(groupe_peu_ambigu)
]

print(f"Changements (cas peu ambigus) : {len(changements_peu_ambigus)}")
changements_peu_ambigus[colonnes_affichage]

Changements (cas peu ambigus) : 1397


,uuid,entity,classe,source,date,vote_result,wikidata_url,wikidata_url_valide
24,c935a0f2-a518-4964-9242-34b847464d31,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
25,87c9b2b6-b99e-4d2f-bf6c-b3a3d1e019b7,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
27,74f241d3-3049-4310-ba54-d0d21d32939e,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
31,1a806393-2098-4207-9917-f19bdfda3a7a,Lacroix,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q3120001
33,c35bf626-20fe-4453-8ad6-f5316aa7bab0,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
...,...,...,...,...,...,...,...,...
4619,c6f05c3a-2aaa-4b44-a5c5-68528a1e85b7,le rapporteur,SPK,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q2861822
4625,6d9f5b5a-163e-4adc-907d-b09bbf5b0f74,le ministre de la guerre de la Restauration,PER,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,NIL
4647,c93c9bc6-7b46-4ac2-8cf3-9c19f673bbf6,le rapporteur,PER,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q2861822
4672,a0d27fe8-37ec-4840-9170-ea8f5ae69e1e,le rapporteur,SPK,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q2861822


In [ ]:
# Répartition par valeur exacte de vote_result à l'intérieur du groupe
changements_peu_ambigus["vote_result"].value_counts()

,count
vote_result,
resolved_unique,1015
resolved_consensus,382


### Cas très ambigus : ambiguous et unresolved

In [ ]:
groupe_tres_ambigu = ["ambiguous", "unresolved"]

changements_tres_ambigus = df[
    df["url_changed"] & df["vote_result"].isin(groupe_tres_ambigu)
]

print(f"Changements (cas très ambigus) : {len(changements_tres_ambigus)}")
changements_tres_ambigus[colonnes_affichage]

changements_tres_ambigus["vote_result"].value_counts()

Changements (cas très ambigus) : 280


,count
vote_result,
ambiguous,172
unresolved,108


### Cas intra-séance

In [ ]:
changements_intra_seance = df[
    df["url_changed"] & (df["vote_result"] == "intra-seance")
]

print(f"Changements (intra-séance) : {len(changements_intra_seance)}")
changements_intra_seance[colonnes_affichage]

Changements (intra-séance) : 0


,uuid,entity,classe,source,date,vote_result,wikidata_url,wikidata_url_valide


### Cas NIL → https (ajout d'une URL)

In [ ]:
nil_vers_url = df[
    (df["url_avant_norm"].isna()) & (df["url_apres_norm"].notna())
]

print(f"Cas NIL -> URL : {len(nil_vers_url)}")
nil_vers_url[colonnes_affichage]

Cas NIL -> URL : 1460


,uuid,entity,classe,source,date,vote_result,wikidata_url,wikidata_url_valide
24,c935a0f2-a518-4964-9242-34b847464d31,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
25,87c9b2b6-b99e-4d2f-bf6c-b3a3d1e019b7,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
27,74f241d3-3049-4310-ba54-d0d21d32939e,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
31,1a806393-2098-4207-9917-f19bdfda3a7a,Lacroix,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q3120001
33,c35bf626-20fe-4453-8ad6-f5316aa7bab0,Edouard Lockroy,PER,ark:/12148/bpt6k64574218,1890-06-24,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q733836
...,...,...,...,...,...,...,...,...
4673,c2f2e846-5eab-4b6c-8c3e-efae1b33b132,le rapporteur,SPK,ark:/12148/bpt6k6383360c,1889-05-21,resolved_consensus,NIL,https://www.wikidata.org/wiki/Q2861822
4674,014b5df8-fae8-4d4d-83fa-b06e8e637e1a,le général Robert,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q3386817
4675,6f5ed8a1-9905-452b-8b39-8b58dafcdc09,le président,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q585025
4676,b10de028-5bef-4d85-8db0-978a9721d0cf,le président,SPK,ark:/12148/bpt6k6383360c,1889-05-21,unresolved,NIL,https://www.wikidata.org/wiki/Q585025


### Cas https → NIL (suppression de l'URL)




In [ ]:
url_vers_nil = df[
    (df["url_avant_norm"].notna()) & (df["url_apres_norm"].isna())
]

print(f"Cas URL -> NIL : {len(url_vers_nil)}")
url_vers_nil[colonnes_affichage]

Cas URL -> NIL : 25


,uuid,entity,classe,source,date,vote_result,wikidata_url,wikidata_url_valide
34,7acebac6-2a42-4eb2-b9f8-ec3e98df2409,le marquis de La Ferrière,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q3131576,NIL
103,35fdf45f-cd90-429e-94b8-31e837ae4c2e,secrétaire général,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q2612305,NIL
106,06f41181-2693-45f0-bccb-358197970fb4,le secrétaire général,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q2612305,NIL
115,e8c765bd-91bd-495c-8b0f-a5759b57e3ef,le maréchal des logis,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q16026042,NIL
116,94d09bf2-f3a7-4c80-98bd-ae0fa2c94a58,le secrétaire général,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q2612305,NIL
119,d3b6cae6-d88d-42ec-a332-342b8a3572a3,le secrétaire général,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q2612305,NIL
205,39b41b8e-2cc7-494f-8bfe-fc82b55ff6ca,le maréchal des logis,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q16026042,NIL
219,b680f7af-2358-4692-8b36-2d96e73294fa,le maréchal des logis de gendarmerie,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q16026042,NIL
235,7c3edaf2-8b55-4c55-ab52-cdbbbd19b04a,le secrétaire général,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q2612305,NIL
244,9aa2b346-8465-477f-be7b-ae8c87cd9729,le maréchal des logis de gendarmerie,PER,ark:/12148/bpt6k64574218,1890-06-24,ambiguous,https://www.wikidata.org/wiki/Q16026042,NIL


In [ ]:
# Détail des types de changement : vide -> valeur, valeur -> vide, valeur -> autre valeur
def type_changement(row):
    avant, apres = row["url_avant_norm"], row["url_apres_norm"]
    if avant == apres:
        return "inchangé"
    if avant is None and apres is not None:
        return "ajout (vide -> valeur)"
    if avant is not None and apres is None:
        return "suppression (valeur -> vide)"
    return "correction (valeur -> autre valeur)"

df["type_changement_url"] = df.apply(type_changement, axis=1)

pd.crosstab(df["vote_result"], df["type_changement_url"], dropna=False, margins=True)

type_changement_url,ajout (vide -> valeur),correction (valeur -> autre valeur),inchangé,suppression (valeur -> vide),All
vote_result,,,,,
ambiguous,26,40,1283,24,1373
intra-seance,0,0,130,0,130
resolved_consensus,349,1,1709,1,2060
resolved_unique,977,0,38,0,1015
unresolved,108,0,0,0,108
All,1460,41,3160,25,4686


### Notes
- Le fichier est lu avec `sep="\t"` (CSV tabulé). Adaptez le chemin `votre_fichier.csv`.
- Si `vote_result` contient lui-même des valeurs vides, elles apparaissent comme une catégorie à part grâce à `dropna=False`.
- `applymap` peut être lent sur de très gros fichiers ; remplacez si besoin par une boucle colonne par colonne.